In [1]:
!pip install -U google-genai pandas numpy tqdm python-dotenv json-repair

In [2]:
from __future__ import annotations

import os
import re
import gc
import json
import time
import uuid
import pickle
import hashlib
import datetime as dt
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from json_repair import repair_json

from google import genai
from google.genai import types

load_dotenv()

# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
# assert GEMINI_API_KEY, "Missing GEMINI_API_KEY or GOOGLE_API_KEY."

client = genai.Client(api_key="")

PROVIDER = "gemini"
MODEL_NAME = "gemini-2.5-pro"
EXPERIMENT_ID = "gemini_followup_simplestrat5_g2css3"

TEMPERATURE = 1.0
PLANNING_TEMPERATURE = 0.0

# Match the old Gemini data collection pipeline.
THINKING_LEVEL = None
THINKING_BUDGET = 128
INCLUDE_THINKING_CONFIG_IN_BATCH = True
GEMINI_UPLOAD_MIME_TYPE = "jsonl"

N_FINAL_PER_TASK_METHOD_STRATEGY = 150
N_STRATA = 5
N_PER_STRATUM = N_FINAL_PER_TASK_METHOD_STRATEGY // N_STRATA
assert N_STRATA * N_PER_STRATUM == N_FINAL_PER_TASK_METHOD_STRATEGY

MAX_OUTPUT_TOKENS_BY_FAMILY = {
    "slogan": 512,
    "aut": 768,
    "story": 2048,
}

EXPERIMENT_DATA_PATH = Path("final_analysis/input_data/experiment_data.pkl")

RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]

DATA_ROOT = (
    Path("ai_data")
    / "deflect_creativity"
    / PROVIDER
    / f"model_{MODEL_NAME}"
    / EXPERIMENT_ID
    / f"run_{RUN_ID}"
)

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "baseline": DATA_ROOT / "01_loaded_baseline",
    "planning_plans": DATA_ROOT / "02_simplestrat_planning" / "plans",
    "planning_batch_inputs": DATA_ROOT / "02_simplestrat_planning" / "batch_inputs",
    "planning_uploaded_files": DATA_ROOT / "02_simplestrat_planning" / "uploaded_files",
    "planning_manifests": DATA_ROOT / "02_simplestrat_planning" / "manifests",
    "planning_raw_outputs": DATA_ROOT / "02_simplestrat_planning" / "raw_outputs",
    "planning_parsed": DATA_ROOT / "02_simplestrat_planning" / "parsed",
    "g2_processing": DATA_ROOT / "03_g2_css_processing",
    "round2_plans": DATA_ROOT / "04_round2_new_baselines" / "plans",
    "round2_batch_inputs": DATA_ROOT / "04_round2_new_baselines" / "batch_inputs",
    "round2_uploaded_files": DATA_ROOT / "04_round2_new_baselines" / "uploaded_files",
    "round2_manifests": DATA_ROOT / "04_round2_new_baselines" / "manifests",
    "round2_raw_outputs": DATA_ROOT / "04_round2_new_baselines" / "raw_outputs",
    "round2_parsed": DATA_ROOT / "04_round2_new_baselines" / "parsed",
    "compiled": DATA_ROOT / "05_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run directory:")
print(DATA_ROOT)

/Users/raiyanabdulbaten/miniforge3/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/raiyanabdulbaten/miniforge3/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


Run directory:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707


In [3]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def json_safe(obj: Any) -> Any:
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)):
            return None
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        val = float(obj)
        return None if (np.isnan(val) or np.isinf(val)) else val
    if isinstance(obj, np.ndarray):
        return [json_safe(x) for x in obj.tolist()]
    try:
        if pd.isna(obj) and not isinstance(obj, (list, tuple, dict)):
            return None
    except Exception:
        pass
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    return str(obj)


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_safe(obj), f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(json_safe(record), ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def safe_slug(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: str, n: int = 24) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip()
    text = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", text)
    text = re.sub(r"\s*```$", "", text).strip()
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()
    return text


def normalize_embeddings(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float64)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms


def first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise RuntimeError(f"None of these candidate columns found: {candidates}\nAvailable columns:\n{df.columns.tolist()}")


def to_jsonable(obj: Any) -> Any:
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    try:
        return json.loads(json.dumps(obj, default=str))
    except Exception:
        return str(obj)


def enum_name(x: Any) -> str:
    if x is None:
        return ""
    if hasattr(x, "name"):
        return x.name
    return str(x)

In [4]:
TASK_SETTINGS = [
    {"task_id": "slogan_smartphone", "task_family": "slogan", "task_label": "Smartphone slogan", "task_prompt_key": "smartphone"},
    {"task_id": "slogan_soda", "task_family": "slogan", "task_label": "Soda slogan", "task_prompt_key": "soda"},
    {"task_id": "slogan_blood_donation", "task_family": "slogan", "task_label": "Blood donation slogan", "task_prompt_key": "blood_donation"},
    {"task_id": "aut_shoe", "task_family": "aut", "task_label": "AUT: shoe", "task_prompt_key": "shoe", "object": "shoe", "common_use": "used as footwear"},
    {"task_id": "aut_button", "task_family": "aut", "task_label": "AUT: button", "task_prompt_key": "button", "object": "button", "common_use": "used to fasten things"},
    {"task_id": "aut_key", "task_family": "aut", "task_label": "AUT: key", "task_prompt_key": "key", "object": "key", "common_use": "used to open a lock"},
    {"task_id": "aut_wooden_pencil", "task_family": "aut", "task_label": "AUT: wooden pencil", "task_prompt_key": "wooden_pencil", "object": "wooden pencil", "common_use": "used for writing"},
    {"task_id": "aut_automobile_tire", "task_family": "aut", "task_label": "AUT: automobile tire", "task_prompt_key": "automobile_tire", "object": "automobile tire", "common_use": "used on the wheel of an automobile"},
    {"task_id": "story_jungle", "task_family": "story", "task_label": "Story: jungle", "task_prompt_key": "jungle"},
    {"task_id": "story_parachute", "task_family": "story", "task_label": "Story: parachute", "task_prompt_key": "parachute"},
    {"task_id": "story_horror", "task_family": "story", "task_label": "Story: horror", "task_prompt_key": "horror"},
    {"task_id": "story_life_last_seconds", "task_family": "story", "task_label": "Story: life / last seconds", "task_prompt_key": "life_last_seconds"},
]

TASK_BY_ID = {t["task_id"]: t for t in TASK_SETTINGS}
TASK_ORDER = [t["task_id"] for t in TASK_SETTINGS]
STRATEGIES = ["vanilla", "diverge"]

SYSTEM_INSTRUCTIONS = (
    "You are participating in a controlled creativity experiment. "
    "Follow the task instructions exactly. Return exactly one response. "
    "Do not explain your reasoning. Do not include commentary before or after the response."
)


def strategy_block(strategy: str) -> str:
    if strategy == "vanilla":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task."
        )
    if strategy == "diverge":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task.\n"
            "- Try to make it stand out from other responses that might be generated for this same task."
        )
    raise ValueError(f"Unknown strategy: {strategy}")


def base_task_prompt(task: dict) -> str:
    task_id = task["task_id"]

    if task_id == "slogan_smartphone":
        return (
            "You are part of the marketing team at a tech company preparing to launch a new smartphone.\n\n"
            "Generate exactly one marketing slogan for this brand-new smartphone.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the smartphone.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_soda":
        return (
            "You are part of the marketing team at a beverage company preparing to launch a new soda.\n\n"
            "Generate exactly one marketing slogan for this brand-new soda.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the soda.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_blood_donation":
        return (
            "You are part of the communications team at a nonprofit organization preparing a campaign "
            "to encourage blood donation.\n\n"
            "Generate exactly one campaign slogan for this blood donation campaign.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the campaign.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id in {"aut_shoe", "aut_button", "aut_key", "aut_wooden_pencil", "aut_automobile_tire"}:
        return (
            "You are participating in a creativity task.\n\n"
            f"Object: {task['object']}\n"
            f"Common use to avoid: {task['common_use']}\n\n"
            "Generate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\n"
            "Requirements:\n"
            "- Do not use the common use.\n"
            "- Do not list multiple uses.\n"
            "- The response must be written in English.\n"
            "- Return only the alternative use as a short phrase or one sentence."
        )

    if task_id == "story_jungle":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story about an adventure in the jungle.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_parachute":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story based on this prompt:\n"
            "The parachute isn’t opening up.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_horror":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one short horror story designed to chill the bones.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_life_last_seconds":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story in 8 sentences. The first sentence must describe 100 years of a character's life. "
            "The next 7 sentences must describe the last 10 seconds of that character's life.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not number or label the sentences.\n"
            "- Do not state which sentence does what.\n"
            "- Return only the story as one paragraph."
        )

    raise ValueError(f"Unknown task_id: {task_id}")


def round2_final_line_for_context(strategy: str, context_type: str) -> str:
    if strategy == "vanilla":
        return "Now generate one new response for the same task."

    if strategy == "diverge" and context_type == "prior_responses":
        return (
            "Now generate one new response for the same task. "
            "It should stand out from the previous response(s) shown above while still satisfying all task requirements."
        )

    if strategy == "diverge" and context_type == "stratum":
        return (
            "Now generate one new response for the same task. "
            "It should stand out from other responses that might be generated for this same task while still satisfying all task requirements."
        )

    raise ValueError(f"Unknown strategy/context_type: {strategy}, {context_type}")

In [5]:
run_config = {
    "experiment_id": EXPERIMENT_ID,
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "temperature": TEMPERATURE,
    "planning_temperature": PLANNING_TEMPERATURE,
    "thinking_level": THINKING_LEVEL,
    "thinking_budget": THINKING_BUDGET,
    "include_thinking_config_in_batch": INCLUDE_THINKING_CONFIG_IN_BATCH,
    "gemini_upload_mime_type": GEMINI_UPLOAD_MIME_TYPE,
    "n_final_per_task_method_strategy": N_FINAL_PER_TASK_METHOD_STRATEGY,
    "n_strata": N_STRATA,
    "n_per_stratum": N_PER_STRATUM,
    "task_order": TASK_ORDER,
    "strategies": STRATEGIES,
    "data_root": str(DATA_ROOT),
    "experiment_data_path": str(EXPERIMENT_DATA_PATH),
    "created_at_utc": now_iso(),
}

config_path = DIRS["metadata"] / f"experiment_config__{RUN_ID}.json"
write_json(config_path, run_config)

config_path

PosixPath('ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/00_metadata/experiment_config__20260523_135949__dddf5707.json')

In [6]:
@dataclass
class ExperimentData:
    long_df: Optional[pd.DataFrame] = None
    embeddings: Optional[np.ndarray] = None


@dataclass
class ProviderExperimentData:
    long_df: Optional[pd.DataFrame] = None
    embeddings: Optional[np.ndarray] = None
    provider: Optional[str] = None
    provider_label: Optional[str] = None
    model: Optional[str] = None


class GenericPicklePlaceholder:
    def __init__(self, *args, **kwargs):
        self.__dict__.update(kwargs)

    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)
        else:
            self.__dict__["state"] = state


class CompatibleUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "__main__":
            if name == "ExperimentData":
                return ExperimentData
            if name == "ProviderExperimentData":
                return ProviderExperimentData
            return GenericPicklePlaceholder
        return super().find_class(module, name)


def load_experiment_data(path: Path) -> ExperimentData:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Run the final analysis setup first, or update EXPERIMENT_DATA_PATH."
        )

    with open(path, "rb") as f:
        obj = CompatibleUnpickler(f).load()

    if hasattr(obj, "long_df") and hasattr(obj, "embeddings") and obj.long_df is not None and obj.embeddings is not None:
        return obj

    candidate_attrs = getattr(obj, "__dict__", {})
    provider_objects = []

    for attr_value in candidate_attrs.values():
        if isinstance(attr_value, dict):
            provider_objects.extend(
                [
                    v for v in attr_value.values()
                    if hasattr(v, "long_df") and hasattr(v, "embeddings")
                    and v.long_df is not None and v.embeddings is not None
                ]
            )
        elif isinstance(attr_value, (list, tuple)):
            provider_objects.extend(
                [
                    v for v in attr_value
                    if hasattr(v, "long_df") and hasattr(v, "embeddings")
                    and v.long_df is not None and v.embeddings is not None
                ]
            )

    if provider_objects:
        long_parts = [p.long_df for p in provider_objects]
        emb_parts = [np.asarray(p.embeddings) for p in provider_objects]
        combined = ExperimentData(
            long_df=pd.concat(long_parts, ignore_index=True, sort=False),
            embeddings=np.vstack(emb_parts),
        )
        if len(combined.long_df) != combined.embeddings.shape[0]:
            raise RuntimeError("Concatenated long_df and embeddings row counts do not match.")
        return combined

    raise RuntimeError("Loaded pickle object does not contain usable long_df and embeddings.")


experiment_data = load_experiment_data(EXPERIMENT_DATA_PATH)

long_df = experiment_data.long_df.copy()
long_df["_row_pos"] = np.arange(len(long_df))

if len(long_df) != experiment_data.embeddings.shape[0]:
    raise RuntimeError(
        f"long_df rows and embeddings rows do not match: {len(long_df)} vs {experiment_data.embeddings.shape[0]}"
    )

TEXT_COL = first_existing_col(
    long_df,
    ["text", "response_text", "output_text", "model_output", "clean_text", "idea", "response", "text_clean"],
)

print("Loaded experiment data:")
print("long_df:", long_df.shape)
print("embeddings:", experiment_data.embeddings.shape)
print("Using text column:", TEXT_COL)
print("Providers:", sorted(long_df["provider"].astype(str).unique()))

Loaded experiment data:
long_df: (64800, 53)
embeddings: (64800, 768)
Using text column: text
Providers: ['anthropic', 'gemini', 'openai']


In [7]:
baseline_mask = (
    long_df["provider"].astype(str).eq(PROVIDER)
    & long_df["round"].astype(int).eq(1)
    & long_df["strategy"].astype(str).eq("vanilla")
    & long_df["condition"].astype(str).eq("base")
    & long_df["task_id"].astype(str).isin(TASK_ORDER)
)

baseline_df = long_df.loc[baseline_mask].copy()

baseline_df["baseline_text"] = baseline_df[TEXT_COL].map(clean_model_text)
baseline_df = baseline_df.sort_values(["task_id", "group_id", "agent_id"]).reset_index(drop=True)

counts = (
    baseline_df
    .groupby("task_id", observed=True)
    .agg(
        n=("baseline_text", "size"),
        n_nonempty=("baseline_text", lambda x: x.notna().sum()),
        task_family=("task_family", "first"),
    )
    .reset_index()
    .sort_values("task_id")
)

display(counts)

missing_tasks = sorted(set(TASK_ORDER) - set(counts["task_id"]))
bad_counts = counts[counts["n"] != N_FINAL_PER_TASK_METHOD_STRATEGY]

if missing_tasks:
    raise RuntimeError(f"Missing tasks in baseline data: {missing_tasks}")

if not bad_counts.empty:
    raise RuntimeError(f"Expected 150 baseline rows per task. Bad counts:\n{bad_counts}")

if baseline_df["baseline_text"].isna().any() or baseline_df["baseline_text"].eq("").any():
    bad = baseline_df[baseline_df["baseline_text"].isna() | baseline_df["baseline_text"].eq("")]
    raise RuntimeError(f"Empty baseline text rows found:\n{bad.head()}")

baseline_light_cols = [
    "_row_pos", "provider", "provider_label", "model", "round", "task_id", "task_label",
    "task_family", "task_family_label", "strategy", "condition", "group_id", "agent_id",
    "agent_index", "baseline_text"
]
baseline_light_cols = [c for c in baseline_light_cols if c in baseline_df.columns]

baseline_light = baseline_df[baseline_light_cols].copy()

baseline_pkl_path = DIRS["baseline"] / "gemini_neutral_r1_base_12tasks_150each.pkl"
baseline_csv_path = DIRS["baseline"] / "gemini_neutral_r1_base_12tasks_150each.csv"

baseline_light.to_pickle(baseline_pkl_path)
baseline_light.to_csv(baseline_csv_path, index=False)

print("Saved:")
print(baseline_pkl_path)
print(baseline_csv_path)

,task_id,n,n_nonempty,task_family
0,slogan_smartphone,150,150,slogan
1,slogan_soda,150,150,slogan
2,slogan_blood_donation,150,150,slogan
3,aut_shoe,150,150,aut
4,aut_button,150,150,aut
5,aut_key,150,150,aut
6,aut_wooden_pencil,150,150,aut
7,aut_automobile_tire,150,150,aut
8,story_jungle,150,150,story
9,story_parachute,150,150,story


Saved:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/01_loaded_baseline/gemini_neutral_r1_base_12tasks_150each.pkl
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/01_loaded_baseline/gemini_neutral_r1_base_12tasks_150each.csv


In [8]:
PLANNING_SYSTEM_INSTRUCTIONS = (
    "You identify semantic diversity strata for controlled text-generation experiments. "
    "Return valid JSON only. Do not include markdown fences or commentary."
)


def build_simplestrat_planning_prompt(task: dict) -> str:
    return f"""
We will later generate 150 independent responses to the following task.

Task prompt:
{base_task_prompt(task)}

Identify exactly {N_STRATA} mutually distinct semantic strata for valid responses to this task.

Use the following procedure internally before choosing the final strata:
1. Consider questions that would separate the space of possible valid responses into broad, meaningfully different groups.
2. Prefer distinctions that would split the possible valid responses into reasonably balanced groups, rather than isolating rare edge cases.
3. Convert the best distinctions into categorical conceptual directions for generation.
4. Exclude distinctions based only on superficial wording, tone, length, punctuation, formatting, or synonyms.
5. Exclude strata that name a specific candidate answer, force a specific phrase, or make the original task harder to satisfy.

The final strata must satisfy all of these requirements:
- Each stratum must be a semantic/content direction, not a superficial style change.
- Each stratum must be broad enough to support many different valid responses.
- The strata must be mutually distinct enough that responses generated under different strata are likely to differ conceptually.
- The strata must collectively cover a wide range of plausible valid responses to the task.
- The generation_instruction must be concise and usable as an added constraint in a later generation prompt.

Return JSON only with this exact structure:
{{
  "task_id": "...",
  "strata": [
    {{
      "stratum_id": 1,
      "name": "short name",
      "description": "one sentence describing the semantic direction",
      "generation_instruction": "a concise instruction that can be appended to the generation prompt",
      "why_broad": "one short sentence explaining why this stratum can support many valid responses",
      "why_distinct": "one short sentence explaining how this stratum differs from the other strata"
    }}
  ]
}}

The JSON must contain exactly {N_STRATA} strata with stratum_id values 1 through {N_STRATA}.
Do not include any text outside the JSON.
""".strip()


def build_planning_plan() -> pd.DataFrame:
    rows = []
    for task in TASK_SETTINGS:
        request_basis = {
            "provider": PROVIDER,
            "model": MODEL_NAME,
            "experiment_id": EXPERIMENT_ID,
            "stage": "simplestrat_planning",
            "task_id": task["task_id"],
            "task_family": task["task_family"],
            "task_label": task["task_label"],
            "n_strata_requested": N_STRATA,
        }
        request_key = "simplestrat_plan__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

        rows.append({
            **request_basis,
            "request_key": request_key,
            "system_instructions": PLANNING_SYSTEM_INSTRUCTIONS,
            "user_prompt": build_simplestrat_planning_prompt(task),
            "temperature": PLANNING_TEMPERATURE,
            "max_output_tokens": 1600,
            "thinking_level": THINKING_LEVEL,
            "thinking_budget": THINKING_BUDGET,
            "include_thinking_config_in_batch": INCLUDE_THINKING_CONFIG_IN_BATCH,
            "created_at_utc": now_iso(),
        })

    out = pd.DataFrame(rows)
    if out["request_key"].duplicated().any():
        raise RuntimeError("Duplicate planning request_key detected.")
    return out


planning_plan_df = build_planning_plan()

planning_plan_path = DIRS["planning_plans"] / f"simplestrat5_planning_plan__{RUN_ID}.csv"
planning_plan_df.to_csv(planning_plan_path, index=False)

print("Planning requests:", len(planning_plan_df))
display(planning_plan_df[["task_id", "task_family", "request_key"]])
print("Saved:", planning_plan_path)

Planning requests: 12


,task_id,task_family,request_key
0,slogan_smartphone,slogan,simplestrat_plan__62451d3993bea7708601ec44
1,slogan_soda,slogan,simplestrat_plan__96c09e38e994a322a34e59ec
2,slogan_blood_donation,slogan,simplestrat_plan__937e34e849cbced4dbe696ed
3,aut_shoe,aut,simplestrat_plan__ad638993cabd6c425ea45408
4,aut_button,aut,simplestrat_plan__551e849a7e63c4538ce4598b
5,aut_key,aut,simplestrat_plan__4aae5442e7df0329d851318c
6,aut_wooden_pencil,aut,simplestrat_plan__1c7c331397280c954a2cab99
7,aut_automobile_tire,aut,simplestrat_plan__69ad0da1c7decfe4ac02fcdd
8,story_jungle,story,simplestrat_plan__f48b11957d4c135a8a820779
9,story_parachute,story,simplestrat_plan__ad630ee23c1258f3fe43b26b


Saved: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/plans/simplestrat5_planning_plan__20260523_135949__dddf5707.csv


In [9]:
def make_thinking_config_for_batch() -> dict:
    if THINKING_LEVEL is not None and THINKING_BUDGET is not None:
        raise ValueError("Use either THINKING_LEVEL or THINKING_BUDGET, not both.")

    if THINKING_LEVEL is not None:
        return {"thinking_level": THINKING_LEVEL}

    if THINKING_BUDGET is not None:
        return {"thinking_budget": int(THINKING_BUDGET)}

    return {}


def make_gemini_generate_content_request(row: pd.Series) -> dict:
    generation_config = {
        "temperature": float(row["temperature"]),
        "max_output_tokens": int(row["max_output_tokens"]),
        "candidate_count": 1,
        "response_modalities": ["TEXT"],
    }

    thinking_config = make_thinking_config_for_batch()

    if bool(row.get("include_thinking_config_in_batch", INCLUDE_THINKING_CONFIG_IN_BATCH)) and thinking_config:
        generation_config["thinking_config"] = thinking_config

    return {
        "contents": [
            {
                "role": "user",
                "parts": [{"text": row["user_prompt"]}],
            }
        ],
        "system_instruction": {
            "parts": [{"text": row["system_instructions"]}]
        },
        "generation_config": generation_config,
    }


def upload_gemini_batch_file(batch_jsonl_path: Path, display_name: str) -> dict:
    uploaded_file = client.files.upload(
        file=str(batch_jsonl_path),
        config=types.UploadFileConfig(
            display_name=display_name,
            mime_type=GEMINI_UPLOAD_MIME_TYPE,
        ),
    )
    return to_jsonable(uploaded_file)


def make_gemini_batch_jsonl_from_plan(
    plan_df: pd.DataFrame,
    stage_name: str,
    batch_input_dir: Path,
    plan_dir: Path,
    stem_prefix: str,
) -> tuple[Path, Path]:
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{stem_prefix}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{timestamp}"

    plan_path = plan_dir / f"{stem}__plan.csv"
    jsonl_path = batch_input_dir / f"{stem}__batch_input.jsonl"

    plan_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    if plan_path.exists() or jsonl_path.exists():
        raise FileExistsError("Refusing to overwrite existing plan or JSONL file.")

    plan_df.to_csv(plan_path, index=False)

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            record = {
                "key": row["request_key"],
                "request": make_gemini_generate_content_request(row),
            }
            f.write(json.dumps(json_safe(record), ensure_ascii=False) + "\n")

    print(f"Wrote plan:  {plan_path}")
    print(f"Wrote JSONL: {jsonl_path}")
    print(f"Requests:    {len(plan_df):,}")

    preview = read_jsonl(jsonl_path)[0]
    print("\nFirst JSONL record preview:")
    print(json.dumps(preview, ensure_ascii=False, indent=2)[:4000])

    return jsonl_path, plan_path


def submit_gemini_batch_generic(
    batch_jsonl_path: Path,
    stage_name: str,
    plan_path: Path,
    manifest_dir: Path,
    uploaded_files_dir: Path,
    display_name_prefix: str,
) -> dict:
    display_name = f"{display_name_prefix}__{EXPERIMENT_ID}__{stage_name}__{MODEL_NAME}__{RUN_ID}"

    uploaded_dump = upload_gemini_batch_file(
        batch_jsonl_path=batch_jsonl_path,
        display_name=display_name,
    )

    uploaded_path = uploaded_files_dir / f"{stage_name}__uploaded_file__{safe_slug(display_name)}.json"

    if uploaded_path.exists():
        raise FileExistsError(f"Refusing to overwrite uploaded-file manifest: {uploaded_path}")

    write_json(uploaded_path, uploaded_dump)

    uploaded_name = uploaded_dump.get("name")
    assert uploaded_name, f"Could not find uploaded file name in uploaded_dump: {uploaded_dump}"

    batch_job = client.batches.create(
        model=MODEL_NAME,
        src=uploaded_name,
        config={"display_name": display_name},
    )

    batch_dump = to_jsonable(batch_job)

    batch_info = {
        "run_id": RUN_ID,
        "experiment_id": EXPERIMENT_ID,
        "stage": stage_name,
        "provider": PROVIDER,
        "model": MODEL_NAME,
        "batch_job": batch_dump,
        "batch_name": batch_dump.get("name"),
        "state_at_submission": batch_dump.get("state"),
        "submitted_at_utc": now_iso(),
        "uploaded_file": uploaded_dump,
        "uploaded_file_manifest": str(uploaded_path),
        "batch_jsonl_path": str(batch_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = (
        manifest_dir
        / f"{EXPERIMENT_ID}__{stage_name}__{PROVIDER}__{MODEL_NAME}__batch_manifest__{stable_hash(batch_info['batch_name'], 16)}.json"
    )

    if manifest_path.exists():
        raise FileExistsError(f"Refusing to overwrite batch manifest: {manifest_path}")

    write_json(manifest_path, batch_info)
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted Gemini batch:")
    print(json.dumps(json_safe(batch_info), indent=2))

    return batch_info


def check_gemini_batch(batch_name: str) -> dict:
    batch_job = client.batches.get(name=batch_name)
    info = to_jsonable(batch_job)
    print(json.dumps(json_safe(info), indent=2))
    return info


def find_result_file_name_from_batch_dump(batch_dump: dict) -> Optional[str]:
    for parent_key in ["dest", "output", "response"]:
        parent = batch_dump.get(parent_key) or {}
        for key in ["fileName", "file_name", "name"]:
            value = parent.get(key)
            if isinstance(value, str) and value.startswith("files/"):
                return value

    def walk(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k in {"fileName", "file_name", "name"} and isinstance(v, str) and v.startswith("files/"):
                    return v
                found = walk(v)
                if found:
                    return found
        elif isinstance(obj, list):
            for item in obj:
                found = walk(item)
                if found:
                    return found
        return None

    return walk(batch_dump)


def download_gemini_batch_results_generic(
    batch_name: str,
    raw_output_dir: Path,
    stage_name: str,
) -> Optional[Path]:
    batch_job = client.batches.get(name=batch_name)
    batch_dump = to_jsonable(batch_job)

    state = enum_name(getattr(batch_job, "state", None)) or str(batch_dump.get("state", ""))

    if "SUCCEEDED" not in state:
        print(f"Batch has not succeeded yet. Current state: {state}")
        if batch_dump.get("error"):
            print("Batch error:")
            print(json.dumps(json_safe(batch_dump.get("error")), indent=2))
        return None

    result_file_name = find_result_file_name_from_batch_dump(batch_dump)

    if not result_file_name:
        print("Could not find result file name. Batch dump preview:")
        print(json.dumps(json_safe(batch_dump), indent=2)[:5000])
        raise ValueError("Could not find result file name in batch object.")

    output_path = raw_output_dir / f"{EXPERIMENT_ID}__{stage_name}__{stable_hash(batch_name, 16)}__results.jsonl"

    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output file: {output_path}")

    file_content = client.files.download(file=result_file_name)

    if isinstance(file_content, bytes):
        content_bytes = file_content
    elif hasattr(file_content, "read"):
        content_bytes = file_content.read()
    else:
        content_bytes = str(file_content).encode("utf-8")

    output_path.write_bytes(content_bytes)

    print(f"Downloaded results to: {output_path}")
    return output_path

In [10]:
def extract_text_from_gemini_response(response: dict) -> str:
    if not isinstance(response, dict):
        return ""

    if response.get("text"):
        return str(response["text"]).strip()

    candidates = response.get("candidates") or []
    texts = []

    for cand in candidates:
        content = cand.get("content") or {}
        parts = content.get("parts") or []

        for part in parts:
            if not isinstance(part, dict):
                continue
            if part.get("thought") is True:
                continue
            if "text" in part and part.get("text") is not None:
                texts.append(str(part.get("text", "")))

    return "\n".join(texts).strip()


def flatten_gemini_usage_metadata(response: dict) -> dict:
    usage = response.get("usage_metadata") or response.get("usageMetadata") or {}

    return {
        "usage_prompt_token_count": usage.get("prompt_token_count") or usage.get("promptTokenCount"),
        "usage_candidates_token_count": usage.get("candidates_token_count") or usage.get("candidatesTokenCount"),
        "usage_thoughts_token_count": usage.get("thoughts_token_count") or usage.get("thoughtsTokenCount"),
        "usage_cached_content_token_count": usage.get("cached_content_token_count") or usage.get("cachedContentTokenCount"),
        "usage_total_token_count": usage.get("total_token_count") or usage.get("totalTokenCount"),
        "usage_raw": usage,
    }


def extract_finish_reason_from_gemini_response(response: dict) -> Optional[str]:
    candidates = response.get("candidates") or []
    if not candidates:
        return None
    return candidates[0].get("finish_reason") or candidates[0].get("finishReason")


def extract_gemini_response_body_and_error(rec: dict) -> tuple[Optional[dict], Optional[Any]]:
    if not isinstance(rec, dict):
        return None, rec

    if rec.get("error") is not None:
        return None, rec.get("error")

    response = rec.get("response")

    if response is None:
        return None, rec.get("status") or rec

    if not isinstance(response, dict):
        return None, response

    if response.get("error") is not None:
        return None, response.get("error")

    if response.get("body") is not None:
        body = response.get("body")
        if isinstance(body, dict) and body.get("error") is not None:
            return None, body.get("error")
        return body, None

    if response.get("response") is not None:
        body = response.get("response")
        if isinstance(body, dict) and body.get("error") is not None:
            return None, body.get("error")
        return body, None

    if (
        response.get("candidates") is not None
        or response.get("usageMetadata") is not None
        or response.get("usage_metadata") is not None
        or response.get("text") is not None
    ):
        return response, None

    return None, response


def parse_gemini_batch_output_to_standard_df(
    batch_output_path: Path,
    plan_path: Path,
    batch_name: str,
) -> pd.DataFrame:
    plan_df = pd.read_csv(plan_path)
    plan_by_key = {row["request_key"]: row.to_dict() for _, row in plan_df.iterrows()}

    batch_records = read_jsonl(batch_output_path)
    parsed_records = []

    for rec in batch_records:
        request_key = rec.get("key") or rec.get("metadata", {}).get("key") or rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})

        response_body, error = extract_gemini_response_body_and_error(rec)

        if response_body is not None:
            text = clean_model_text(extract_text_from_gemini_response(response_body))
            usage_flat = flatten_gemini_usage_metadata(response_body)
            finish_reason = extract_finish_reason_from_gemini_response(response_body)
            status = "success" if text else "empty_text"

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": status,
                "text": text,
                "provider_response_id": None,
                "finish_reason": finish_reason,
                "usage_prompt_token_count": usage_flat["usage_prompt_token_count"],
                "usage_candidates_token_count": usage_flat["usage_candidates_token_count"],
                "usage_thoughts_token_count": usage_flat["usage_thoughts_token_count"],
                "usage_cached_content_token_count": usage_flat["usage_cached_content_token_count"],
                "usage_total_token_count": usage_flat["usage_total_token_count"],
                "usage": usage_flat["usage_raw"],
                "error": None if text else "No visible final-answer text extracted from Gemini response.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_name": batch_name,
                "raw_result_type": "response",
                "raw_record": rec,
            }
        else:
            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "finish_reason": None,
                "usage_prompt_token_count": None,
                "usage_candidates_token_count": None,
                "usage_thoughts_token_count": None,
                "usage_cached_content_token_count": None,
                "usage_total_token_count": None,
                "usage": None,
                "error": error,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_name": batch_name,
                "raw_result_type": "error",
                "raw_record": rec,
            }

        parsed_records.append(record)

    return pd.DataFrame(parsed_records)


def save_parsed_df(
    parsed_df: pd.DataFrame,
    parsed_dir: Path,
    stage_name: str,
    batch_name: str,
    prefix: str,
) -> dict:
    batch_hash = stable_hash(batch_name, 16)

    parsed_jsonl_path = parsed_dir / f"{prefix}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{batch_hash}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"{prefix}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{batch_hash}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"{prefix}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{batch_hash}__parsed.pkl"

    if parsed_jsonl_path.exists() or parsed_csv_path.exists() or parsed_pkl_path.exists():
        raise FileExistsError("Refusing to overwrite existing parsed files.")

    for _, row in parsed_df.iterrows():
        append_jsonl(parsed_jsonl_path, row.to_dict())

    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "experiment_id": EXPERIMENT_ID,
        "stage": stage_name,
        "batch_name": batch_name,
        "n_records": len(parsed_df),
        "n_success": int(parsed_df["status"].eq("success").sum()),
        "n_empty_text": int(parsed_df["status"].eq("empty_text").sum()),
        "n_error": int(parsed_df["status"].eq("error").sum()),
        "n_max_token_finish": int(
            parsed_df["finish_reason"]
            .astype(str)
            .str.upper()
            .str.contains("MAX|TOKEN", regex=True, na=False)
            .sum()
        ),
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"{prefix}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{batch_hash}__parse_summary.json"
    write_json(summary_path, summary)

    print(json.dumps(json_safe(summary), indent=2))
    return summary

In [11]:
planning_jsonl_path, planning_plan_path_saved = make_gemini_batch_jsonl_from_plan(
    plan_df=planning_plan_df,
    stage_name="simplestrat5_planning",
    batch_input_dir=DIRS["planning_batch_inputs"],
    plan_dir=DIRS["planning_plans"],
    stem_prefix=EXPERIMENT_ID,
)

planning_batch_info = submit_gemini_batch_generic(
    batch_jsonl_path=planning_jsonl_path,
    stage_name="simplestrat5_planning",
    plan_path=planning_plan_path_saved,
    manifest_dir=DIRS["planning_manifests"],
    uploaded_files_dir=DIRS["planning_uploaded_files"],
    display_name_prefix="deflect_creativity_followup",
)

planning_batch_info

Wrote plan:  ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/plans/gemini_followup_simplestrat5_g2css3__simplestrat5_planning__gemini__gemini-2.5-pro__20260523_140141__plan.csv
Wrote JSONL: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/batch_inputs/gemini_followup_simplestrat5_g2css3__simplestrat5_planning__gemini__gemini-2.5-pro__20260523_140141__batch_input.jsonl
Requests:    12

First JSONL record preview:
{
  "key": "simplestrat_plan__62451d3993bea7708601ec44",
  "request": {
    "contents": [
      {
        "role": "user",
        "parts": [
          {
            "text": "We will later generate 150 independent responses to the following task.\n\nTask prompt:\nYou are part of the marketing team at a tech company preparing to launch a new smartphone.\n\nGenerate exactly one marketing slo

{'run_id': '20260523_135949__dddf5707',
 'experiment_id': 'gemini_followup_simplestrat5_g2css3',
 'stage': 'simplestrat5_planning',
 'provider': 'gemini',
 'model': 'gemini-2.5-pro',
 'batch_job': {'name': 'batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p',
  'display_name': 'deflect_creativity_followup__gemini_followup_simplestrat5_g2css3__simplestrat5_planning__gemini-2.5-pro__20260523_135949__dddf5707',
  'state': 'JOB_STATE_PENDING',
  'error': None,
  'create_time': '2026-05-23T18:01:43.014501Z',
  'start_time': None,
  'end_time': None,
  'update_time': '2026-05-23T18:01:43.014501Z',
  'model': 'models/gemini-2.5-pro',
  'src': None,
  'dest': None},
 'batch_name': 'batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p',
 'state_at_submission': 'JOB_STATE_PENDING',
 'submitted_at_utc': '2026-05-23T18:01:43.567156+00:00',
 'uploaded_file': {'name': 'files/424464wn51l7',
  'display_name': 'deflect_creativity_followup__gemini_followup_simplestrat5_g2css3__simplestrat5_planning__gemini-2.5-pro__2

In [13]:
planning_status = check_gemini_batch(planning_batch_info["batch_name"])

{
  "name": "batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p",
  "display_name": "deflect_creativity_followup__gemini_followup_simplestrat5_g2css3__simplestrat5_planning__gemini-2.5-pro__20260523_135949__dddf5707",
  "state": "JOB_STATE_SUCCEEDED",
  "error": null,
  "create_time": "2026-05-23T18:01:43.014501Z",
  "start_time": null,
  "end_time": "2026-05-23T18:02:36.071476Z",
  "update_time": "2026-05-23T18:02:36.071476Z",
  "model": "models/gemini-2.5-pro",
  "src": null,
  "dest": {
    "format": null,
    "gcs_uri": null,
    "bigquery_uri": null,
    "file_name": "files/batch-hz6ett6lui0okrofetxtunw717jgym0j4i9p",
    "inlined_responses": null,
    "inlined_embed_content_responses": null
  }
}


In [14]:
planning_output_path = download_gemini_batch_results_generic(
    batch_name=planning_batch_info["batch_name"],
    raw_output_dir=DIRS["planning_raw_outputs"],
    stage_name="simplestrat5_planning",
)

planning_df = parse_gemini_batch_output_to_standard_df(
    batch_output_path=planning_output_path,
    plan_path=Path(planning_batch_info["plan_path"]),
    batch_name=planning_batch_info["batch_name"],
)

planning_parse_summary = save_parsed_df(
    parsed_df=planning_df,
    parsed_dir=DIRS["planning_parsed"],
    stage_name="simplestrat5_planning",
    batch_name=planning_batch_info["batch_name"],
    prefix=EXPERIMENT_ID,
)

display(planning_df["status"].value_counts(dropna=False))
display(planning_df[["task_id", "status", "finish_reason", "text", "error"]])

Downloaded results to: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/raw_outputs/gemini_followup_simplestrat5_g2css3__simplestrat5_planning__155802b1cfe63433__results.jsonl
{
  "experiment_id": "gemini_followup_simplestrat5_g2css3",
  "stage": "simplestrat5_planning",
  "batch_name": "batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p",
  "n_records": 12,
  "n_success": 12,
  "n_empty_text": 0,
  "n_error": 0,
  "n_max_token_finish": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/parsed/gemini_followup_simplestrat5_g2css3__simplestrat5_planning__gemini__gemini-2.5-pro__155802b1cfe63433__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/parsed

status
success    12
Name: count, dtype: int64

,task_id,status,finish_reason,text,error
0,slogan_smartphone,success,STOP,"{\n ""task_id"": ""smartphone-slogan-strata-v1"",...",None
1,slogan_soda,success,STOP,"{\n ""task_id"": ""soda-slogan-strata-v1"",\n ""s...",None
2,slogan_blood_donation,success,STOP,"{\n ""task_id"": ""blood_donation_slogan_01"",\n ...",None
3,aut_shoe,success,STOP,"{\n ""task_id"": ""shoe_alternative_use_strata_1...",None
4,aut_button,success,STOP,"{\n ""task_id"": ""button_alternative_use_strata...",None
5,aut_key,success,STOP,"{\n ""task_id"": ""key_alternative_use_01"",\n ""...",None
6,aut_wooden_pencil,success,STOP,"{\n ""task_id"": ""pencil_alt_use_01"",\n ""strat...",None
7,aut_automobile_tire,success,STOP,"{\n ""task_id"": ""tire_alternative_use_01"",\n ...",None
8,story_jungle,success,STOP,"{\n ""task_id"": ""jungle_adventure_story_v1"",\n...",None
9,story_parachute,success,STOP,"{\n ""task_id"": ""parachute_story_strata_v1"",\n...",None


In [15]:
def extract_first_balanced_json_object(text: str) -> str:
    s = str(text).strip()
    start = s.find("{")
    if start == -1:
        raise ValueError(f"No opening brace found:\n{s[:1000]}")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(s)):
        ch = s[i]

        if escape:
            escape = False
            continue

        if ch == "\\":
            escape = True
            continue

        if ch == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return s[start:i + 1]

    raise ValueError(f"Could not find balanced JSON object:\n{s[:1500]}")


def parse_planning_json_robust(text: str) -> tuple[dict, str]:
    raw = str(text).strip()
    raw = re.sub(r"^```(?:json)?", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw).strip()

    try:
        return json.loads(raw), "strict_full"
    except Exception:
        pass

    candidate = extract_first_balanced_json_object(raw)

    try:
        return json.loads(candidate), "strict_balanced_object"
    except Exception:
        pass

    repaired_obj = repair_json(candidate, return_objects=True)

    if isinstance(repaired_obj, dict):
        return repaired_obj, "json_repair_object"

    if isinstance(repaired_obj, str):
        return json.loads(repaired_obj), "json_repair_string"

    raise ValueError(f"Could not parse repaired object for text:\n{raw[:2000]}")


def build_strata_table_from_planning_robust(parsed_pkl_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    planning_df = pd.read_pickle(parsed_pkl_path)

    rows = []
    audit_rows = []

    required_fields = [
        "stratum_id",
        "name",
        "description",
        "generation_instruction",
        "why_broad",
        "why_distinct",
    ]

    for _, row in planning_df.iterrows():
        if row["status"] != "success":
            raise RuntimeError(f"Planning row failed: {row.to_dict()}")

        task_id = row["task_id"]
        text = row["text"]

        try:
            obj, parse_method = parse_planning_json_robust(text)
            parse_error = None
            parse_status = "parsed"
        except Exception as e:
            obj = None
            parse_method = None
            parse_error = repr(e)
            parse_status = "failed"

        audit_rows.append({
            "task_id": task_id,
            "request_key": row["request_key"],
            "parse_status": parse_status,
            "parse_method": parse_method,
            "parse_error": parse_error,
            "raw_text": text,
            "parsed_object_json": json.dumps(obj, ensure_ascii=False) if obj is not None else None,
        })

        if obj is None:
            continue

        strata = obj.get("strata")

        if not isinstance(strata, list):
            raise RuntimeError(f"No strata list found for task={task_id}:\n{obj}")

        if len(strata) != N_STRATA:
            raise RuntimeError(f"Expected {N_STRATA} strata for task={task_id}, got {len(strata)}:\n{obj}")

        seen_ids = []
        for s in strata:
            for field in required_fields:
                if not str(s.get(field, "")).strip():
                    raise RuntimeError(f"Missing or empty field `{field}` for task={task_id}:\n{s}")

            sid = int(s.get("stratum_id"))
            seen_ids.append(sid)

            rows.append({
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "experiment_id": EXPERIMENT_ID,
                "task_id": task_id,
                "task_family": row["task_family"],
                "task_label": row["task_label"],
                "stratum_id": sid,
                "stratum_name": str(s.get("name", "")).strip(),
                "stratum_description": str(s.get("description", "")).strip(),
                "stratum_generation_instruction": str(s.get("generation_instruction", "")).strip(),
                "why_broad": str(s.get("why_broad", "")).strip(),
                "why_distinct": str(s.get("why_distinct", "")).strip(),
                "planning_request_key": row["request_key"],
                "planning_text": text,
                "planning_parse_method": parse_method,
                "planning_batch_name": row["batch_name"],
                "created_at_utc": now_iso(),
            })

        if sorted(seen_ids) != list(range(1, N_STRATA + 1)):
            raise RuntimeError(f"Bad stratum ids for task={task_id}: {seen_ids}")

    audit_df = pd.DataFrame(audit_rows)

    failed = audit_df[audit_df["parse_status"].eq("failed")]
    if len(failed) > 0:
        audit_path = DIRS["planning_parsed"] / "simplestrat5_planning_parse_failures.csv"
        failed.to_csv(audit_path, index=False)
        display(failed[["task_id", "request_key", "parse_error"]])
        raise RuntimeError(f"{len(failed)} planning rows could not be parsed. Saved failures to: {audit_path}")

    out = pd.DataFrame(rows).sort_values(["task_id", "stratum_id"]).reset_index(drop=True)
    return out, audit_df


strata_df, planning_parse_audit_df = build_strata_table_from_planning_robust(
    Path(planning_parse_summary["parsed_pkl_path"])
)

strata_csv_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.csv"
strata_pkl_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.pkl"
strata_json_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.json"
planning_audit_csv_path = DIRS["planning_parsed"] / "simplestrat5_planning_parse_audit.csv"

strata_df.to_csv(strata_csv_path, index=False)
strata_df.to_pickle(strata_pkl_path)
write_json(strata_json_path, strata_df.to_dict(orient="records"))
planning_parse_audit_df.to_csv(planning_audit_csv_path, index=False)

print("Saved strata:")
print(strata_csv_path)
print(strata_pkl_path)
print(strata_json_path)
print("Saved parse audit:")
print(planning_audit_csv_path)

display(planning_parse_audit_df.groupby(["parse_method"], dropna=False).size().reset_index(name="n"))
display(strata_df)

Saved strata:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/parsed/simplestrat5_validated_strata.csv
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/parsed/simplestrat5_validated_strata.pkl
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/parsed/simplestrat5_validated_strata.json
Saved parse audit:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/02_simplestrat_planning/parsed/simplestrat5_planning_parse_audit.csv


,parse_method,n
0,strict_full,12


,provider,model,experiment_id,task_id,task_family,task_label,stratum_id,stratum_name,stratum_description,stratum_generation_instruction,why_broad,why_distinct,planning_request_key,planning_text,planning_parse_method,planning_batch_name,created_at_utc
0,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,1,Recreational or Play Equipment,The response describes a use for the tire as a...,"Generate a use related to recreation, play, or...","There are many different types of games, sport...",This stratum focuses on human leisure and ente...,simplestrat_plan__69ad0da1c7decfe4ac02fcdd,"{\n ""task_id"": ""tire_alternative_use_01"",\n ...",strict_full,batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p,2026-05-23T18:03:56.532057+00:00
1,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,2,Gardening or Landscaping,The response describes a use for the tire in a...,"Generate a use related to gardening, farming, ...","Tires can be used as planters, borders, retain...","This stratum is specific to outdoor, horticult...",simplestrat_plan__69ad0da1c7decfe4ac02fcdd,"{\n ""task_id"": ""tire_alternative_use_01"",\n ...",strict_full,batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p,2026-05-23T18:03:56.532065+00:00
2,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,3,Furniture or Household Item,The response describes a use for the tire as a...,Generate a use as a piece of furniture or a fu...,Tires can be transformed into many different t...,This stratum focuses on creating functional do...,simplestrat_plan__69ad0da1c7decfe4ac02fcdd,"{\n ""task_id"": ""tire_alternative_use_01"",\n ...",strict_full,batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p,2026-05-23T18:03:56.532073+00:00
3,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,4,Artistic or Decorative Object,The response describes a use for the tire as a...,"Generate a use as a sculpture, art installatio...",The tire's shape and material can be the basis...,This stratum's primary goal is aesthetic expre...,simplestrat_plan__69ad0da1c7decfe4ac02fcdd,"{\n ""task_id"": ""tire_alternative_use_01"",\n ...",strict_full,batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p,2026-05-23T18:03:56.532081+00:00
4,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,5,Structural or Safety Material,The response describes a use for the tire as a...,"Generate a use as a building material, a safet...",The durability and shock-absorbent properties ...,"This stratum involves large-scale, functional ...",simplestrat_plan__69ad0da1c7decfe4ac02fcdd,"{\n ""task_id"": ""tire_alternative_use_01"",\n ...",strict_full,batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p,2026-05-23T18:03:56.532089+00:00
5,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,1,Miniature Tool or Component,The response describes using the button as a f...,Generate a use where the button acts as a mini...,"The button can be imagined as a wheel, gear, s...",This stratum focuses on the button's physical ...,simplestrat_plan__551e849a7e63c4538ce4598b,"{\n ""task_id"": ""button_alternative_use_strata...",strict_full,batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p,2026-05-23T18:03:56.531724+00:00
6,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,2,Artistic or Decorative Medium,The response describes using the button as a c...,Generate a use where the button is a component...,"Buttons can be used to create mosaics, jewelry...",This stratum focuses on the button's aesthetic...,simplestrat_plan__551e849a7e63c4538ce4598b,"{\n ""task_id"": ""button_alternative_use_strata...",strict_full,batches/hz6ett6lui0okrofetxtunw717jgym0j4i9p,2026-05-23T18:03:56.531732+00:00
7,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,3,Game Piece or To

In [16]:
def select_css_medoid_farthest(X: np.ndarray, k: int = 3) -> list[int]:
    if X.shape[0] < k:
        raise ValueError(f"Need at least {k} rows, got {X.shape[0]}")

    Xn = normalize_embeddings(X)
    sim = np.clip(Xn @ Xn.T, -1.0, 1.0)
    dist = 1.0 - sim

    avg_dist = dist.mean(axis=1)
    selected = [int(np.argmin(avg_dist))]

    while len(selected) < k:
        remaining = [i for i in range(X.shape[0]) if i not in selected]
        min_dist_to_selected = dist[np.ix_(remaining, selected)].min(axis=1)
        next_idx = remaining[int(np.argmax(min_dist_to_selected))]
        selected.append(int(next_idx))

    return selected


def build_g2_css_anchor_table(
    baseline_df: pd.DataFrame,
    embeddings: np.ndarray,
    k: int = 3,
) -> pd.DataFrame:
    rows = []

    for task_id in TASK_ORDER:
        task_df = (
            baseline_df[baseline_df["task_id"].astype(str).eq(task_id)]
            .copy()
            .sort_values(["group_id", "agent_id"])
            .reset_index(drop=True)
        )

        if len(task_df) != N_FINAL_PER_TASK_METHOD_STRATEGY:
            raise RuntimeError(f"Expected 150 baseline rows for {task_id}, got {len(task_df)}")

        row_pos = task_df["_row_pos"].to_numpy(dtype=int)
        X = embeddings[row_pos]
        selected_local = select_css_medoid_farthest(X, k=k)

        for rank, local_idx in enumerate(selected_local, start=1):
            r = task_df.iloc[local_idx].to_dict()
            rows.append({
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "experiment_id": EXPERIMENT_ID,
                "task_id": task_id,
                "task_family": r.get("task_family"),
                "task_label": r.get("task_label", TASK_BY_ID[task_id]["task_label"]),
                "anchor_rank": rank,
                "selection_rule": "medoid_start_farthest_first_css",
                "baseline_row_pos": int(r["_row_pos"]),
                "baseline_group_id": r.get("group_id"),
                "baseline_agent_id": r.get("agent_id"),
                "anchor_text": clean_model_text(r["baseline_text"]),
                "created_at_utc": now_iso(),
            })

    out = pd.DataFrame(rows).sort_values(["task_id", "anchor_rank"]).reset_index(drop=True)

    counts = out.groupby("task_id").size()
    if not (counts == k).all():
        raise RuntimeError(f"Bad anchor counts:\n{counts}")

    return out


g2_anchors_df = build_g2_css_anchor_table(
    baseline_df=baseline_light,
    embeddings=experiment_data.embeddings,
    k=3,
)

g2_anchor_csv_path = DIRS["g2_processing"] / "g2_css_static3_anchors.csv"
g2_anchor_pkl_path = DIRS["g2_processing"] / "g2_css_static3_anchors.pkl"
g2_anchor_json_path = DIRS["g2_processing"] / "g2_css_static3_anchors.json"

g2_anchors_df.to_csv(g2_anchor_csv_path, index=False)
g2_anchors_df.to_pickle(g2_anchor_pkl_path)
write_json(g2_anchor_json_path, g2_anchors_df.to_dict(orient="records"))

print("Saved G2/CSS anchors:")
print(g2_anchor_csv_path)
print(g2_anchor_pkl_path)
print(g2_anchor_json_path)

display(g2_anchors_df)

Saved G2/CSS anchors:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/03_g2_css_processing/g2_css_static3_anchors.csv
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/03_g2_css_processing/g2_css_static3_anchors.pkl
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/03_g2_css_processing/g2_css_static3_anchors.json


,provider,model,experiment_id,task_id,task_family,task_label,anchor_rank,selection_rule,baseline_row_pos,baseline_group_id,baseline_agent_id,anchor_text,created_at_utc
0,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,1,medoid_start_farthest_first_css,55868,base_035,base_035__a1,Used as an aggregate in shock-absorbent asphal...,2026-05-23T18:04:02.716429+00:00
1,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,2,medoid_start_farthest_first_css,56094,base_148,base_148__a1,"An interlocking, modular base for a floating b...",2026-05-23T18:04:02.716492+00:00
2,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,3,medoid_start_farthest_first_css,55860,base_031,base_031__a1,"A sound-absorbing, textured wall panel for a r...",2026-05-23T18:04:02.716549+00:00
3,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,1,medoid_start_farthest_first_css,50462,base_032,base_032__a1,"Used as textured, non-slip grips on the handle...",2026-05-23T18:04:02.710341+00:00
4,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,2,medoid_start_farthest_first_css,50522,base_062,base_062__a1,Individual mosaic tiles for a detailed art piece.,2026-05-23T18:04:02.710408+00:00
5,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,3,medoid_start_farthest_first_css,50428,base_015,base_015__a1,"A weight for calibrating a small, sensitive sc...",2026-05-23T18:04:02.710464+00:00
6,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,1,medoid_start_farthest_first_css,52492,base_147,base_147__a1,Used as a conductive bridge to complete a simp...,2026-05-23T18:04:02.712373+00:00
7,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,2,medoid_start_farthest_first_css,52440,base_121,base_121__a1,A key's unique bitting pattern is used as a te...,2026-05-23T18:04:02.712460+00:00
8,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,3,medoid_start_farthest_first_css,52356,base_079,base_079__a1,A weight for calibrating a small precision scale.,2026-05-23T18:04:02.712531+00:00
9,gemini,gemini-2.5-pro,gemini_followup_simplestrat5_g2css3,aut_shoe,aut,AUT: shoe,1,medoid_start_farthest_first_css,48768,base_085,base_085__a1,A sturdy shoe heel used as a pestle to grind s...,2026-05-23T18:04:02.708665+00:00


In [17]:
def build_simplestrat_r2_prompt(task: dict, strategy: str, stratum: dict) -> str:
    context = (
        "Conceptual direction assigned for this round:\n"
        f"{stratum['stratum_name']}: {stratum['stratum_description']}\n\n"
        "Additional generation constraint:\n"
        f"{stratum['stratum_generation_instruction']}\n\n"
        "Use this direction as the main conceptual path for the response. "
        "Do not mention the direction label or explain the direction.\n\n"
    )

    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + context
        + round2_final_line_for_context(strategy, context_type="stratum")
    )


def build_g2_css_r2_prompt(task: dict, strategy: str, anchors: pd.DataFrame) -> str:
    anchors = anchors.sort_values("anchor_rank")
    if len(anchors) != 3:
        raise RuntimeError(f"Expected exactly 3 anchors for task={task['task_id']}, got {len(anchors)}")

    context = (
        "Previous responses from three other agents in the same first round:\n"
        f'1. "{anchors.iloc[0]["anchor_text"]}"\n'
        f'2. "{anchors.iloc[1]["anchor_text"]}"\n'
        f'3. "{anchors.iloc[2]["anchor_text"]}"\n\n'
    )

    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + context
        + round2_final_line_for_context(strategy, context_type="prior_responses")
    )


def slot_to_stratum_id(slot_num: int) -> int:
    return ((slot_num - 1) % N_STRATA) + 1


def build_round2_new_baselines_plan(strata_df: pd.DataFrame, g2_anchors_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    strata_lookup = {
        (r.task_id, int(r.stratum_id)): r._asdict()
        for r in strata_df.itertuples(index=False)
    }

    for task in TASK_SETTINGS:
        task_id = task["task_id"]
        task_anchors = g2_anchors_df[g2_anchors_df["task_id"].astype(str).eq(task_id)].copy()

        for method in ["simplestrat5", "g2_css_static3"]:
            for strategy in STRATEGIES:
                for slot_num in range(1, N_FINAL_PER_TASK_METHOD_STRATEGY + 1):
                    slot_id = f"slot_{slot_num:03d}"

                    if method == "simplestrat5":
                        stratum_id = slot_to_stratum_id(slot_num)
                        stratum = strata_lookup[(task_id, stratum_id)]
                        user_prompt = build_simplestrat_r2_prompt(task, strategy, stratum)
                        anchor_count = 0
                        context_count = 1
                        method_label = "SimpleStrat-lite, 5 fixed auto-stratified semantic strata"

                    elif method == "g2_css_static3":
                        stratum_id = None
                        user_prompt = build_g2_css_r2_prompt(task, strategy, task_anchors)
                        anchor_count = 3
                        context_count = 3
                        method_label = "G2-inspired static CSS, 3 representative R1 examples"

                    else:
                        raise ValueError(method)

                    request_basis = {
                        "provider": PROVIDER,
                        "model": MODEL_NAME,
                        "experiment_id": EXPERIMENT_ID,
                        "round": 2,
                        "task_id": task_id,
                        "task_family": task["task_family"],
                        "task_label": task["task_label"],
                        "strategy": strategy,
                        "method": method,
                        "method_label": method_label,
                        "condition": method,
                        "slot_id": slot_id,
                        "slot_num": slot_num,
                        "stratum_id": stratum_id,
                        "anchor_count": anchor_count,
                        "context_count": context_count,
                    }

                    request_key = "r2new__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

                    rows.append({
                        **request_basis,
                        "request_key": request_key,
                        "system_instructions": SYSTEM_INSTRUCTIONS,
                        "user_prompt": user_prompt,
                        "temperature": TEMPERATURE,
                        "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                        "thinking_level": THINKING_LEVEL,
                        "thinking_budget": THINKING_BUDGET,
                        "include_thinking_config_in_batch": INCLUDE_THINKING_CONFIG_IN_BATCH,
                        "created_at_utc": now_iso(),
                    })

    out = pd.DataFrame(rows)

    if out["request_key"].duplicated().any():
        dupes = out[out["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise RuntimeError(f"Duplicate request_key detected:\n{dupes.head()}")

    return out


round2_new_plan_df = build_round2_new_baselines_plan(
    strata_df=strata_df,
    g2_anchors_df=g2_anchors_df,
)

expected_n = len(TASK_SETTINGS) * 2 * len(STRATEGIES) * N_FINAL_PER_TASK_METHOD_STRATEGY
print("Expected R2 new-baseline requests:", expected_n)
print("Actual R2 new-baseline requests:  ", len(round2_new_plan_df))

display(
    round2_new_plan_df
    .groupby(["method", "strategy", "task_id"], observed=True)
    .agg(
        n=("request_key", "size"),
        n_strata=("stratum_id", lambda x: x.dropna().nunique()),
        n_slots=("slot_id", "nunique"),
    )
    .reset_index()
)

round2_plan_csv_path = DIRS["round2_plans"] / f"round2_new_baselines_plan__{RUN_ID}.csv"
round2_plan_pkl_path = DIRS["round2_plans"] / f"round2_new_baselines_plan__{RUN_ID}.pkl"

round2_new_plan_df.to_csv(round2_plan_csv_path, index=False)
round2_new_plan_df.to_pickle(round2_plan_pkl_path)

print("Saved:")
print(round2_plan_csv_path)
print(round2_plan_pkl_path)

Expected R2 new-baseline requests: 7200
Actual R2 new-baseline requests:   7200


,method,strategy,task_id,n,n_strata,n_slots
0,g2_css_static3,diverge,aut_automobile_tire,150,0,150
1,g2_css_static3,diverge,aut_button,150,0,150
2,g2_css_static3,diverge,aut_key,150,0,150
3,g2_css_static3,diverge,aut_shoe,150,0,150
4,g2_css_static3,diverge,aut_wooden_pencil,150,0,150
5,g2_css_static3,diverge,slogan_blood_donation,150,0,150
6,g2_css_static3,diverge,slogan_smartphone,150,0,150
7,g2_css_static3,diverge,slogan_soda,150,0,150
8,g2_css_static3,diverge,story_horror,150,0,150
9,g2_css_static3,diverge,story_jungle,150,0,150


Saved:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/04_round2_new_baselines/plans/round2_new_baselines_plan__20260523_135949__dddf5707.csv
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/04_round2_new_baselines/plans/round2_new_baselines_plan__20260523_135949__dddf5707.pkl


In [18]:
for method in ["simplestrat5", "g2_css_static3"]:
    for strategy in ["vanilla", "diverge"]:
        ex = round2_new_plan_df[
            (round2_new_plan_df["method"] == method)
            & (round2_new_plan_df["strategy"] == strategy)
            & (round2_new_plan_df["task_id"] == "slogan_smartphone")
        ].iloc[0]

        print("\n" + "=" * 120)
        print(method, strategy, ex["request_key"])
        print("=" * 120)
        print(ex["user_prompt"])


simplestrat5 vanilla r2new__c8709c229384ff765835c7a4
You are part of the marketing team at a tech company preparing to launch a new smartphone.

Generate exactly one marketing slogan for this brand-new smartphone.

Requirements:
- The slogan must not exceed 6 words.
- The slogan must be written in English.
- You may assume any detail about the smartphone.
- Do not list multiple slogans.
- Return only the slogan text.

Creativity goal:
- Make the response novel and appropriate for the task.

Conceptual direction assigned for this round:
Focus on a specific feature: The slogan highlights a single, standout technological feature of the smartphone, such as the camera, battery life, or processing speed.

Additional generation constraint:
Generate a slogan that emphasizes a specific, innovative feature of the smartphone.

Use this direction as the main conceptual path for the response. Do not mention the direction label or explain the direction.

Now generate one new response for the same t

In [19]:
round2_jsonl_path, round2_plan_path_saved = make_gemini_batch_jsonl_from_plan(
    plan_df=round2_new_plan_df,
    stage_name="round2_new_baselines",
    batch_input_dir=DIRS["round2_batch_inputs"],
    plan_dir=DIRS["round2_plans"],
    stem_prefix=EXPERIMENT_ID,
)

round2_batch_info = submit_gemini_batch_generic(
    batch_jsonl_path=round2_jsonl_path,
    stage_name="round2_new_baselines",
    plan_path=round2_plan_path_saved,
    manifest_dir=DIRS["round2_manifests"],
    uploaded_files_dir=DIRS["round2_uploaded_files"],
    display_name_prefix="deflect_creativity_followup",
)

round2_batch_info

Wrote plan:  ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/04_round2_new_baselines/plans/gemini_followup_simplestrat5_g2css3__round2_new_baselines__gemini__gemini-2.5-pro__20260523_140413__plan.csv
Wrote JSONL: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/04_round2_new_baselines/batch_inputs/gemini_followup_simplestrat5_g2css3__round2_new_baselines__gemini__gemini-2.5-pro__20260523_140413__batch_input.jsonl
Requests:    7,200

First JSONL record preview:
{
  "key": "r2new__c8709c229384ff765835c7a4",
  "request": {
    "contents": [
      {
        "role": "user",
        "parts": [
          {
            "text": "You are part of the marketing team at a tech company preparing to launch a new smartphone.\n\nGenerate exactly one marketing slogan for this brand-new smartphone.\n\nRequirements:\n- The slogan must not exceed 6 words.\n- The s

{'run_id': '20260523_135949__dddf5707',
 'experiment_id': 'gemini_followup_simplestrat5_g2css3',
 'stage': 'round2_new_baselines',
 'provider': 'gemini',
 'model': 'gemini-2.5-pro',
 'batch_job': {'name': 'batches/v29k9lssduim4c1fhrk6n884edgbbmctgnfq',
  'display_name': 'deflect_creativity_followup__gemini_followup_simplestrat5_g2css3__round2_new_baselines__gemini-2.5-pro__20260523_135949__dddf5707',
  'state': 'JOB_STATE_PENDING',
  'error': None,
  'create_time': '2026-05-23T18:04:18.445472Z',
  'start_time': None,
  'end_time': None,
  'update_time': '2026-05-23T18:04:18.445472Z',
  'model': 'models/gemini-2.5-pro',
  'src': None,
  'dest': None},
 'batch_name': 'batches/v29k9lssduim4c1fhrk6n884edgbbmctgnfq',
 'state_at_submission': 'JOB_STATE_PENDING',
 'submitted_at_utc': '2026-05-23T18:04:20.036379+00:00',
 'uploaded_file': {'name': 'files/x1oq463bwe7u',
  'display_name': 'deflect_creativity_followup__gemini_followup_simplestrat5_g2css3__round2_new_baselines__gemini-2.5-pro__2026

In [21]:
round2_status = check_gemini_batch(round2_batch_info["batch_name"])

{
  "name": "batches/v29k9lssduim4c1fhrk6n884edgbbmctgnfq",
  "display_name": "deflect_creativity_followup__gemini_followup_simplestrat5_g2css3__round2_new_baselines__gemini-2.5-pro__20260523_135949__dddf5707",
  "state": "JOB_STATE_SUCCEEDED",
  "error": null,
  "create_time": "2026-05-23T18:04:18.445472Z",
  "start_time": null,
  "end_time": "2026-05-23T18:06:04.355145Z",
  "update_time": "2026-05-23T18:06:04.355145Z",
  "model": "models/gemini-2.5-pro",
  "src": null,
  "dest": {
    "format": null,
    "gcs_uri": null,
    "bigquery_uri": null,
    "file_name": "files/batch-v29k9lssduim4c1fhrk6n884edgbbmctgnfq",
    "inlined_responses": null,
    "inlined_embed_content_responses": null
  }
}


In [22]:
round2_output_path = download_gemini_batch_results_generic(
    batch_name=round2_batch_info["batch_name"],
    raw_output_dir=DIRS["round2_raw_outputs"],
    stage_name="round2_new_baselines",
)

round2_new_df = parse_gemini_batch_output_to_standard_df(
    batch_output_path=round2_output_path,
    plan_path=Path(round2_batch_info["plan_path"]),
    batch_name=round2_batch_info["batch_name"],
)

round2_parse_summary = save_parsed_df(
    parsed_df=round2_new_df,
    parsed_dir=DIRS["round2_parsed"],
    stage_name="round2_new_baselines",
    batch_name=round2_batch_info["batch_name"],
    prefix=EXPERIMENT_ID,
)

display(round2_new_df["status"].value_counts(dropna=False))
display(round2_new_df["finish_reason"].value_counts(dropna=False).head(20))

Downloaded results to: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/04_round2_new_baselines/raw_outputs/gemini_followup_simplestrat5_g2css3__round2_new_baselines__1c49b4b7a0c4e642__results.jsonl
{
  "experiment_id": "gemini_followup_simplestrat5_g2css3",
  "stage": "round2_new_baselines",
  "batch_name": "batches/v29k9lssduim4c1fhrk6n884edgbbmctgnfq",
  "n_records": 7200,
  "n_success": 7200,
  "n_empty_text": 0,
  "n_error": 0,
  "n_max_token_finish": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/04_round2_new_baselines/parsed/gemini_followup_simplestrat5_g2css3__round2_new_baselines__gemini__gemini-2.5-pro__1c49b4b7a0c4e642__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/04_round2_new_baselines/parse

status
success    7200
Name: count, dtype: int64

finish_reason
STOP    7200
Name: count, dtype: int64

In [23]:
def validate_gemini_round2_new(df: pd.DataFrame, expected_n: int) -> pd.DataFrame:
    print("Expected rows:", expected_n)
    print("Actual rows:  ", len(df))

    display(df["status"].value_counts(dropna=False).reset_index(name="n"))
    display(df["finish_reason"].value_counts(dropna=False).reset_index(name="n").head(20))

    usage_cols = [
        "usage_prompt_token_count",
        "usage_candidates_token_count",
        "usage_thoughts_token_count",
        "usage_cached_content_token_count",
        "usage_total_token_count",
    ]
    existing_usage_cols = [c for c in usage_cols if c in df.columns]

    print("\nToken usage summary:")
    display(df[existing_usage_cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)

    problem_mask = (
        len(df) != expected_n
    )

    bad_df = df[
        ~df["status"].eq("success")
        | df["text"].isna()
        | df["text"].astype(str).str.strip().eq("")
        | df["finish_reason"].astype(str).str.upper().str.contains("MAX|TOKEN", regex=True, na=False)
    ].copy()

    if problem_mask or len(bad_df) > 0:
        print("Bad records:", len(bad_df))
        display(
            bad_df[[
                "request_key",
                "task_id",
                "task_family",
                "strategy",
                "method",
                "slot_id",
                "status",
                "finish_reason",
                "text",
                "error",
            ]].head(100)
        )
        raise RuntimeError("R2 new-baseline Gemini batch has failed/empty/MAX_TOKEN-like records.")

    print("R2 new-baseline Gemini batch passed validation.")
    return bad_df


_ = validate_gemini_round2_new(round2_new_df, expected_n=expected_n)

summary = (
    round2_new_df
    .groupby(["method", "strategy", "task_id"], observed=True)
    .agg(
        n=("text", "size"),
        n_success=("status", lambda x: (x == "success").sum()),
        n_nonempty=("text", lambda x: x.notna().sum()),
        prompt_tokens=("usage_prompt_token_count", "sum"),
        candidate_tokens=("usage_candidates_token_count", "sum"),
        thought_tokens=("usage_thoughts_token_count", "sum"),
        cached_content_tokens=("usage_cached_content_token_count", "sum"),
        total_tokens=("usage_total_token_count", "sum"),
    )
    .reset_index()
)

display(summary)

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

compiled_csv_path = DIRS["compiled"] / f"round2_new_baselines_outputs__{RUN_ID}__{timestamp}.csv"
compiled_pkl_path = DIRS["compiled"] / f"round2_new_baselines_outputs__{RUN_ID}__{timestamp}.pkl"
summary_csv_path = DIRS["compiled"] / f"round2_new_baselines_summary__{RUN_ID}__{timestamp}.csv"

if compiled_csv_path.exists() or compiled_pkl_path.exists() or summary_csv_path.exists():
    raise FileExistsError("Refusing to overwrite compiled output files.")

round2_new_df.to_csv(compiled_csv_path, index=False)
round2_new_df.to_pickle(compiled_pkl_path)
summary.to_csv(summary_csv_path, index=False)

print("Saved compiled outputs:")
print(compiled_csv_path)
print(compiled_pkl_path)
print(summary_csv_path)

Expected rows: 7200
Actual rows:   7200


,status,n
0,success,7200


,finish_reason,n
0,STOP,7200



Token usage summary:


,count,mean,std,min,50%,90%,95%,99%,max
usage_prompt_token_count,7200.0,335.516667,191.831334,191.0,258.0,736.0,776.00,858.0,858.0
usage_candidates_token_count,7200.0,71.690000,82.660219,5.0,19.0,197.0,211.00,232.0,264.0
usage_thoughts_token_count,7200.0,68.857778,15.031441,33.0,67.0,87.0,96.00,119.0,179.0
usage_total_token_count,7200.0,476.064444,258.229666,242.0,350.0,992.0,1046.05,1146.0,1200.0


R2 new-baseline Gemini batch passed validation.


,method,strategy,task_id,n,n_success,n_nonempty,prompt_tokens,candidate_tokens,thought_tokens,cached_content_tokens,total_tokens
0,g2_css_static3,diverge,aut_automobile_tire,150,150,150,39000,2676,12510,0,54186
1,g2_css_static3,diverge,aut_button,150,150,150,37950,2199,10808,0,50957
2,g2_css_static3,diverge,aut_key,150,150,150,41250,3248,10398,0,54896
3,g2_css_static3,diverge,aut_shoe,150,150,150,38700,3253,10444,0,52397
4,g2_css_static3,diverge,aut_wooden_pencil,150,150,150,37950,2556,9882,0,50388
5,g2_css_static3,diverge,slogan_blood_donation,150,150,150,35400,1142,11399,0,47941
6,g2_css_static3,diverge,slogan_smartphone,150,150,150,34800,1069,11691,0,47560
7,g2_css_static3,diverge,slogan_soda,150,150,150,34650,1044,11106,0,46800
8,g2_css_static3,diverge,story_horror,150,150,150,105150,27127,10580,0,142857
9,g2_css_static3,diverge,story_jungle,150,150,150,128700,32742,10307,0,171749


Saved compiled outputs:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/05_compiled/round2_new_baselines_outputs__20260523_135949__dddf5707__20260523_140630.csv
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/05_compiled/round2_new_baselines_outputs__20260523_135949__dddf5707__20260523_140630.pkl
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/05_compiled/round2_new_baselines_summary__20260523_135949__dddf5707__20260523_140630.csv


In [24]:
final_manifest = {
    "run_id": RUN_ID,
    "experiment_id": EXPERIMENT_ID,
    "provider": PROVIDER,
    "model": MODEL_NAME,
    "data_root": str(DATA_ROOT),
    "baseline_pkl_path": str(baseline_pkl_path),
    "baseline_csv_path": str(baseline_csv_path),
    "planning_plan_path": str(planning_plan_path),
    "planning_batch_info": planning_batch_info,
    "planning_parse_summary": planning_parse_summary,
    "strata_csv_path": str(strata_csv_path),
    "strata_pkl_path": str(strata_pkl_path),
    "strata_json_path": str(strata_json_path),
    "g2_anchor_csv_path": str(g2_anchor_csv_path),
    "g2_anchor_pkl_path": str(g2_anchor_pkl_path),
    "g2_anchor_json_path": str(g2_anchor_json_path),
    "round2_plan_csv_path": str(round2_plan_csv_path),
    "round2_plan_pkl_path": str(round2_plan_pkl_path),
    "round2_batch_info": round2_batch_info,
    "round2_parse_summary": round2_parse_summary,
    "compiled_csv_path": str(compiled_csv_path),
    "compiled_pkl_path": str(compiled_pkl_path),
    "summary_csv_path": str(summary_csv_path),
    "created_at_utc": now_iso(),
}

final_manifest_path = DIRS["metadata"] / f"final_manifest__{RUN_ID}.json"
write_json(final_manifest_path, final_manifest)

print("Saved final manifest:")
print(final_manifest_path)

Saved final manifest:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/gemini_followup_simplestrat5_g2css3/run_20260523_135949__dddf5707/00_metadata/final_manifest__20260523_135949__dddf5707.json
